# 04 — Semantic Entropy

Semantic entropy clusters semantically equivalent generations and computes entropy over meanings, not tokens.

## Clustering Semantically Equivalent Generations

A key problem with token-level uncertainty is that different token sequences can express the same meaning. If an LLM generates 'The capital of France is Paris' and 'Paris is France's capital', the token distributions differ but the semantics are identical — this is not uncertainty.

**Semantic entropy** (Kuhn et al., 2023) addresses this:
1. Generate *k* samples (e.g., k=10) from the LLM for the same question
2. **Cluster** the samples by semantic equivalence: use an NLI model (or LLM judge) to determine if two answers entail each other
3. Compute entropy over the clusters (meaning classes) rather than individual token sequences:
$$SE = H(C) = -\sum_c P(c) \log P(c)$$
where *P(c)* is the fraction of samples belonging to cluster *c*.

**Why it works**: if the model is confident, all samples should cluster into one meaning class (SE≈0). If uncertain, samples will fall into multiple classes (high SE).

**Comparison to other uncertainty measures**:
- Semantic entropy outperforms token probability on factual QA tasks
- Requires multiple LLM calls — expensive but more reliable
- Agnostic to paraphrase variation that inflates token-level uncertainty

In [ ]:
# Semantic entropy implementation
import numpy as np
from collections import defaultdict

# Simulated NLI model for semantic equivalence
class SimulatedNLI:
    def entails(self, text1, text2):
        """Returns True if both texts express the same meaning."""
        # Simulate: same question type -> high chance of overlap
        words1 = set(text1.lower().split())
        words2 = set(text2.lower().split())
        overlap = len(words1 & words2) / max(len(words1 | words2), 1)
        return overlap > 0.4  # simplified Jaccard-based equivalence

nli = SimulatedNLI()

def cluster_by_semantic_equivalence(answers, nli_model):
    """Group answers into semantic equivalence classes."""
    clusters = []
    for answer in answers:
        placed = False
        for cluster in clusters:
            # Check if answer is equivalent to cluster representative
            if nli_model.entails(answer, cluster[0]) and nli_model.entails(cluster[0], answer):
                cluster.append(answer)
                placed = True
                break
        if not placed:
            clusters.append([answer])
    return clusters

def semantic_entropy(answers, nli_model):
    """Compute semantic entropy over clustered answers."""
    clusters = cluster_by_semantic_equivalence(answers, nli_model)
    n = len(answers)
    probs = [len(c) / n for c in clusters]
    H = -sum(p * np.log(p + 1e-10) for p in probs)
    return H, clusters

# Demo: confident vs uncertain questions
confident_answers = [
    'Paris is the capital of France.',
    'The capital of France is Paris.',
    'France capital Paris',
    'Paris - capital city of France',
    'Paris is France capital',
]

uncertain_answers = [
    'The answer depends on the context.',
    'I believe it might be around 1850.',
    'Some sources say 1889, others 1887.',
    'It was completed in the late 1880s.',
    'Not entirely sure but probably 1890.',
]

for name, answers in [('Confident', confident_answers), ('Uncertain', uncertain_answers)]:
    H, clusters = semantic_entropy(answers, nli)
    print(f'{name} question:')
    print(f'  Semantic entropy: {H:.3f}')
    print(f'  Clusters: {len(clusters)}')
    for i, c in enumerate(clusters):
        print(f'    Cluster {i+1} ({len(c)} answers): "{c[0][:50]}"')
    print()

In [ ]:
# Semantic entropy vs accuracy correlation
import matplotlib.pyplot as plt

np.random.seed(42)

# Simulate results for 100 questions
def simulate_question(difficulty):
    """Generate k=5 LLM answers and ground truth correctness."""
    is_correct = np.random.random() < difficulty

    if is_correct:
        # Confident: all answers similar
        base = 'The answer is correct option A with value 42'
        answers = [base + ' ' * np.random.randint(0, 5) for _ in range(5)]
    else:
        # Uncertain: diverse answers
        options = [
            'The value is approximately 42',
            'I think the answer might be 17',
            'It depends on the conditions',
            'Some sources suggest 42, others 100',
            'The answer is uncertain and context dependent',
        ]
        answers = list(np.random.choice(options, 5, replace=True))

    H, _ = semantic_entropy(answers, nli)
    return H, int(is_correct)

results = [simulate_question(np.random.uniform(0.3, 0.9)) for _ in range(100)]
se_vals = [r[0] for r in results]
correct_vals = [r[1] for r in results]

# AUROC: can semantic entropy distinguish correct from wrong?
from sklearn.metrics import roc_auc_score
# Low SE -> more likely correct, so use 1-SE
auroc = roc_auc_score(correct_vals, [-se for se in se_vals])
print(f'AUROC (semantic entropy predicting correctness): {auroc:.3f}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.hist([se_vals[i] for i in range(100) if correct_vals[i] == 1],
          bins=20, alpha=0.7, label='Correct', color='steelblue')
ax1.hist([se_vals[i] for i in range(100) if correct_vals[i] == 0],
          bins=20, alpha=0.7, label='Wrong', color='tomato')
ax1.set_xlabel('Semantic Entropy')
ax1.set_ylabel('Count')
ax1.set_title(f'Semantic Entropy by Correctness (AUROC={auroc:.2f})')
ax1.legend()

from sklearn.metrics import roc_curve
fpr, tpr, _ = roc_curve(correct_vals, [-se for se in se_vals])
ax2.plot(fpr, tpr, color='steelblue')
ax2.plot([0,1], [0,1], 'k--')
ax2.set_xlabel('FPR'); ax2.set_ylabel('TPR')
ax2.set_title(f'ROC Curve (AUROC={auroc:.2f})')

plt.tight_layout()
plt.savefig('/tmp/semantic_entropy.png', dpi=100, bbox_inches='tight')
plt.show()
print('Semantic entropy analysis saved')